# 📊 Análise de Inteligência de Vendas em E-commerce com SQL

## 🎯Objetivo
Extrair métricas financeiras e operacionais a partir de um banco de dados relacional para responder a dores de negócio, como:
* Compreensão da evolução mensal de vendas e faturamento.
* Mapeamento de clientes de alto valor para campanhas de retenção.
* Desempenho de produtos e categorias para otimização de estoque.

In [2]:
# Instala a biblioteca para gerar dados realistas
!pip install faker -q

import sqlite3
import pandas as pd
import random
from faker import Faker

# Inicializa o gerador de dados (Faker)
fake = Faker('pt_BR')
Faker.seed(42)
random.seed(42)

# 1. Criação do banco de dados na memória
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Criação das tabelas
cursor.executescript("""
CREATE TABLE clientes (
    cliente_id INTEGER PRIMARY KEY,
    nome TEXT,
    estado TEXT,
    data_cadastro DATE
);

CREATE TABLE produtos (
    produto_id INTEGER PRIMARY KEY,
    nome_produto TEXT,
    categoria TEXT,
    preco REAL
);

CREATE TABLE pedidos (
    pedido_id INTEGER PRIMARY KEY,
    cliente_id INTEGER,
    data_pedido DATE,
    status TEXT,
    FOREIGN KEY (cliente_id) REFERENCES clientes(cliente_id)
);

CREATE TABLE itens_pedido (
    item_id INTEGER PRIMARY KEY,
    pedido_id INTEGER,
    produto_id INTEGER,
    quantidade INTEGER,
    valor_unitario REAL,
    FOREIGN KEY (pedido_id) REFERENCES pedidos(pedido_id),
    FOREIGN KEY (produto_id) REFERENCES produtos(produto_id)
);
""")

# 3. População de dados dinâmica e realista
# Gerando 30 clientes
estados = ['SP', 'RJ', 'MG', 'PR', 'SC', 'RS', 'BA', 'PE']
for i in range(1, 31):
    cursor.execute("INSERT INTO clientes VALUES (?, ?, ?, ?)",
                   (i, fake.name(), random.choice(estados), fake.date_between(start_date='-1y', end_date='today')))

# Gerando 8 produtos de e-commerce
produtos_lista = [
    (101, 'Smartphone Ultra', 'Eletrônicos', 2500.00),
    (102, 'Fone Bluetooth Noise Cancelling', 'Eletrônicos', 450.00),
    (103, 'Smartwatch Sport', 'Eletrônicos', 800.00),
    (104, 'Cafeteira Espresso Automática', 'Eletrodomésticos', 650.00),
    (105, 'Fritadeira Airfryer 4L', 'Eletrodomésticos', 380.00),
    (106, 'Livro: Data Science para Negócios', 'Livros', 95.00),
    (107, 'Livro: Storytelling com Dados', 'Livros', 85.00),
    (108, 'Teclado Mecânico RGB', 'Informática', 320.00)
]
cursor.executemany("INSERT INTO produtos VALUES (?, ?, ?, ?)", produtos_lista)

# Gerando 100 pedidos aleatórios nos últimos 6 meses
status_opcoes = ['Entregue', 'Entregue', 'Entregue', 'Cancelado'] # 75% entregue, 25% cancelado
for i in range(1001, 1101):
    cursor.execute("INSERT INTO pedidos VALUES (?, ?, ?, ?)",
                   (i, random.randint(1, 30), fake.date_between(start_date='-6m', end_date='today'), random.choice(status_opcoes)))

# Gerando itens para cada pedido
item_id_counter = 1
for pedido_id in range(1001, 1101):
    # Quantidade de produtos diferentes no mesmo pedido
    num_itens = random.randint(1, 3)
    produtos_selecionados = random.sample(produtos_lista, num_itens)

    for prod in produtos_selecionados:
        qtd = random.randint(1, 2)
        cursor.execute("INSERT INTO itens_pedido VALUES (?, ?, ?, ?, ?)",
                       (item_id_counter, pedido_id, prod[0], qtd, prod[3]))
        item_id_counter += 1

conn.commit()


# ------------------------------------------------------------------
# QUERY 1: Faturamento Mensal e Crescimento
# ------------------------------------------------------------------
query_faturamento = """
SELECT
    strftime('%Y-%m', p.data_pedido) AS mes,
    COUNT(DISTINCT p.pedido_id) AS total_pedidos,
    ROUND(SUM(i.quantidade * i.valor_unitario), 2) AS faturamento_total
FROM pedidos p
JOIN itens_pedido i ON p.pedido_id = i.pedido_id
WHERE p.status != 'Cancelado'
GROUP BY mes
ORDER BY mes;
"""
df_faturamento = pd.read_sql_query(query_faturamento, conn)


# ------------------------------------------------------------------
# QUERY 2: Top Clientes por Faturamento
# ------------------------------------------------------------------
query_clientes = """
SELECT
    c.nome,
    c.estado,
    COUNT(DISTINCT p.pedido_id) AS total_pedidos,
    ROUND(SUM(i.quantidade * i.valor_unitario), 2) AS total_gasto
FROM clientes c
JOIN pedidos p ON c.cliente_id = p.cliente_id
JOIN itens_pedido i ON p.pedido_id = i.pedido_id
WHERE p.status != 'Cancelado'
GROUP BY c.cliente_id, c.nome, c.estado
ORDER BY total_gasto DESC
LIMIT 5;
"""
df_clientes = pd.read_sql_query(query_clientes, conn)


# ------------------------------------------------------------------
# QUERY 3: Produtos mais vendidos por faturamento
# ------------------------------------------------------------------
query_produtos = """
SELECT
    prod.categoria,
    prod.nome_produto,
    SUM(i.quantidade) AS unidades_vendidas,
    ROUND(SUM(i.quantidade * i.valor_unitario), 2) AS faturamento_produto
FROM produtos prod
JOIN itens_pedido i ON prod.produto_id = i.produto_id
JOIN pedidos p ON i.pedido_id = p.pedido_id
WHERE p.status != 'Cancelado'
GROUP BY prod.categoria, prod.nome_produto
ORDER BY faturamento_produto DESC;
"""
df_produtos = pd.read_sql_query(query_produtos, conn)


# ------------------------------------------------------------------
# EXIBIÇÃO DE RESULTADOS E INSIGHTS DINÂMICOS
# ------------------------------------------------------------------
print("==========================================================")
print("             RELATÓRIO DE INTELIGÊNCIA DE VENDAS          ")
print("==========================================================\n")

print("--- 1. Evolução Mensal de Faturamento ---")
print(df_faturamento.to_string(index=False))
print("\n")

print("--- 2. Top 5 Clientes VIP (Maior Faturamento) ---")
print(df_clientes.to_string(index=False))
print("\n")

print("--- 3. Desempenho de Produtos por Faturamento ---")
print(df_produtos.to_string(index=False))
print("\n")

# Geração automática de insights com base nos dados gerados
melhor_mes = df_faturamento.loc[df_faturamento['faturamento_total'].idxmax()]['mes']
pior_mes = df_faturamento.loc[df_faturamento['faturamento_total'].idxmin()]['mes']
top_cliente = df_clientes.iloc[0]['nome']
top_produto = df_produtos.iloc[0]['nome_produto']
top_categoria = df_produtos.iloc[0]['categoria']

print("==========================================================")
print("                  INSIGHTS AUTOMATIZADOS                  ")
print("==========================================================")
print(f"* O melhor mês de faturamento foi {melhor_mes}, enquanto o menor desempenho foi em {pior_mes}.")
print(f"* O cliente de maior valor para o negócio foi {top_cliente}, liderando em volume de compras.")
print(f"* O produto campeão de faturamento foi o '{top_produto}', consolidando a categoria '{top_categoria}' como estratégica.")
print("==========================================================")

# Fechando a conexão
conn.close()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.0 MB/s eta 0:00:00
             RELATÓRIO DE INTELIGÊNCIA DE VENDAS          

--- 1. Evolução Mensal de Faturamento ---
    mes  total_pedidos  faturamento_total
2026-05             79           135760.0


--- 2. Top 5 Clientes VIP (Maior Faturamento) ---
                  nome estado  total_pedidos  total_gasto
       Thomas Monteiro     PR              4      15120.0
       Juliana Azevedo     MG              4      14470.0
Maria Cecília Oliveira     PR              5      11235.0
        Brayan Pereira     PR              5      10405.0
         Amanda Novais     PR              4      10140.0


--- 3. Desempenho de Produtos por Faturamento ---
       categoria                      nome_produto  unidades_vendidas  faturamento_produto
     Eletrônicos                  Smartphone Ultra                 25              62500.0
     Eletrônicos                  Smartwatch Sport                 27              21600.0
Eletrodomést

/tmp/ipykernel_2075/3767081205.py:57: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("INSERT INTO clientes VALUES (?, ?, ?, ?)",
/tmp/ipykernel_2075/3767081205.py:76: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("INSERT INTO pedidos VALUES (?, ?, ?, ?)",


## Conclusões e Insights do Projeto

1. **Evolução de Faturamento:** Conseguimos mapear a flutuação do faturamento mês a mês. Isso permite planejar melhor as promoções em períodos de baixa e prever demandas de estoque em meses de pico.
2. **Segmentação de Clientes:** Identificamos os clientes que mais geram receita. Essa informação é vital para criar programas de fidelidade ou ofertas personalizadas.
3. **Mix de Produtos:** O relatório por categoria aponta quais produtos têm o maior volume de vendas, direcionando os esforços de marketing e reabastecimento.